In [3]:
"""
Q-G-CTGAN: Quality-Aware Cluster-Conditioned Oversampling
via Intra-Cluster Synthetic Sample Filtering

Notebook 04b: New Baseline Method (ctdGAN)

Addresses Reviewer #1 (R1-2) and Reviewer #2 (R2-2): ctdGAN was
discussed in Related Work but missing from the experimental comparison.

Environment: venv_ctdgan (uses the artsyn package, official
implementation by the ctdGAN authors: lakritidis/ARTSyn, v0.5.2).

ctdGAN (Douzas et al., 2025, arXiv:2508.00472): jointly conditions
generation on both class labels and cluster assignments via
probabilistic subspace sampling. cluster_method='gmm' is used here
(rather than the package default 'kmeans') to align the clustering
mechanism with Q-G-CTGAN and G-CTGAN (both GMM-based), isolating the
comparison to the generation/conditioning strategy itself rather than
conflating it with a difference in clustering algorithm.

NOTE: results from this notebook (04b_ctdgan_results.csv) are combined
with 04a (K-means CTGAN, CTGAN-MOS) and 04c (CTAB-GAN+) results in
Notebook 05 for a unified comparison table, since each requires a
separate, mutually incompatible virtual environment.
"""

import os
import time
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

DATASET_DIR = "./datasets"
RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATASET_NAMES = [
    "credit_default", "fraud_detection", "pima_diabetes", "ibm_attrition",
    "yeast_me2", "mammography", "abalone_19", "wine_quality",
    "ecoli", "pageblocks", "protein_homo",
    "satellite", "churn", "secom", "thyroid_sick", "unsw_nb15",
]

CLASSIFIERS = ["RF", "LGBM", "MLP"]
CTDGAN_EPOCHS = 100  # matches CTGAN epochs used elsewhere (Notebooks 03, 04a)

print(f"Datasets    : {len(DATASET_NAMES)}")
print(f"Classifiers : {CLASSIFIERS}")
print(f"ctdGAN epochs: {CTDGAN_EPOCHS}")


# ════════════════════════════════════════════════════════════
## 1. Classifiers & Evaluation (identical to Notebooks 02/03/04a)
# ════════════════════════════════════════════════════════════

from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    average_precision_score, balanced_accuracy_score, confusion_matrix,
)
from lightgbm import LGBMClassifier


def get_classifier(name, random_state=RANDOM_STATE):
    if name == "RF":
        return RandomForestClassifier(
            n_estimators=200, max_depth=10, min_samples_leaf=3,
            class_weight="balanced", random_state=random_state, n_jobs=-1,
        )
    elif name == "LGBM":
        return LGBMClassifier(
            n_estimators=100, learning_rate=0.05, num_leaves=31,
            class_weight="balanced", random_state=random_state,
            n_jobs=-1, verbose=-1,
        )
    elif name == "MLP":
        return MLPClassifier(
            hidden_layer_sizes=(128, 64), alpha=0.001,
            max_iter=300, random_state=random_state,
        )


def g_mean_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return float(np.sqrt(sensitivity * specificity))


def evaluate(model, X_test, y_test):
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    return {
        "AUC"         : round(roc_auc_score(y_test, y_prob), 4),
        "PR_AUC"      : round(average_precision_score(y_test, y_prob), 4),
        "F1"          : round(f1_score(y_test, y_pred, average="macro", zero_division=0), 4),
        "Precision"   : round(precision_score(y_test, y_pred, average="macro", zero_division=0), 4),
        "Recall"      : round(recall_score(y_test, y_pred, average="macro", zero_division=0), 4),
        "G_mean"      : round(g_mean_score(y_test, y_pred), 4),
        "Balanced_Acc": round(balanced_accuracy_score(y_test, y_pred), 4),
    }

print("Classifiers & Evaluation ready.")


# ════════════════════════════════════════════════════════════
## 2. ctdGAN wrapper
# ════════════════════════════════════════════════════════════

from artsyn.generators.ctd_gan import ctdGAN


def apply_ctdgan(X_train, y_train, epochs=CTDGAN_EPOCHS, random_state=RANDOM_STATE):
    """
    Thin wrapper around ctdGAN.fit_resample for consistency with the
    experiment loop pattern used across Notebooks 02-04. cluster_method
    is fixed to 'gmm' to align with Q-G-CTGAN/G-CTGAN's clustering
    mechanism (see module docstring).
    """
    model = ctdGAN(
        epochs=epochs,
        random_state=random_state,
        cluster_method="gmm",
        sampling_strategy="auto",  # balance minority classes to match majority
    )
    X_res, y_res = model.fit_resample(X_train, y_train)
    return X_res, y_res

print("ctdGAN wrapper ready.")


# ════════════════════════════════════════════════════════════
## 3. Main Experiment Loop
# ════════════════════════════════════════════════════════════
# Same 70/30 stratified split and StandardScaler protocol as Notebooks
# 02/03/04a, for direct comparability. Incremental save after each
# dataset (large datasets, especially unsw_nb15 and fraud_detection,
# can take a long time given ctdGAN's per-cluster GAN training).

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

results = []
out_path = os.path.join(RESULTS_DIR, "04b_ctdgan_results.csv")

for ds_name in DATASET_NAMES:
    path = os.path.join(DATASET_DIR, f"{ds_name}.csv")
    if not os.path.exists(path):
        print(f"[SKIP] {ds_name}: file not found")
        continue

    df = pd.read_csv(path)

    # Cast bool (one-hot encoded) columns to float64 -- same fix applied
    # in Notebooks 02/03/04a for downstream numeric operations.
    bool_cols = df.select_dtypes(include="bool").columns
    if len(bool_cols):
        df[bool_cols] = df[bool_cols].astype("float64")

    X = df.drop(columns=["target"]).values.astype(float)
    y = df["target"].values.astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    print(f"\n{'='*65}")
    print(f"Dataset : {ds_name}  |  n_train={len(X_train):,}  |  minority={y_train.mean():.2%}")
    print(f"{'='*65}")

    t0 = time.time()
    try:
        X_res, y_res = apply_ctdgan(X_train.copy(), y_train.copy())
        gen_time = round(time.time() - t0, 2)
        print(f"  Generated: n={len(y_res):,}  minority={y_res.mean():.2%}  [{gen_time:.1f}s]")
    except Exception as e:
        print(f"  ctdGAN failed: {e}")
        continue

    for clf_name in CLASSIFIERS:
        clf = get_classifier(clf_name)
        t1 = time.time()
        try:
            clf.fit(X_res, y_res)
            train_time = round(time.time() - t1, 2)
            metrics = evaluate(clf, X_test, y_test)
        except Exception as e:
            print(f"    {clf_name} - Failed: {e}")
            continue

        results.append({
            "dataset": ds_name, "method": "ctdGAN", "classifier": clf_name,
            "generation_time": gen_time, "train_time": train_time,
            "n_synthetic": len(y_res) - len(y_train), **metrics,
        })
        print(f"    {clf_name:<5} | AUC={metrics['AUC']:.4f}  "
              f"PR-AUC={metrics['PR_AUC']:.4f}  G-mean={metrics['G_mean']:.4f}")

    pd.DataFrame(results).to_csv(out_path, index=False)
    print(f"\n  === {ds_name} complete; results saved incrementally ===")

print("\n04b experiment complete.")

Datasets    : 16
Classifiers : ['RF', 'LGBM', 'MLP']
ctdGAN epochs: 100
Classifiers & Evaluation ready.
ctdGAN wrapper ready.

Dataset : credit_default  |  n_train=21,000  |  minority=22.12%


ctdGAN Sampling     : 100%|██████████| 2/2 [00:01<00:00,  1.68it/s]


  Generated: n=32,710  minority=50.00%  [737.5s]
    RF    | AUC=0.7557  PR-AUC=0.5328  G-mean=0.5863
    LGBM  | AUC=0.7755  PR-AUC=0.5357  G-mean=0.5883
    MLP   | AUC=0.7036  PR-AUC=0.3984  G-mean=0.5650

  === credit_default complete; results saved incrementally ===

Dataset : fraud_detection  |  n_train=199,364  |  minority=0.17%


ctdGAN Sampling     : 100%|██████████| 2/2 [00:16<00:00,  8.26s/it]


  Generated: n=398,040  minority=50.00%  [7664.0s]
    RF    | AUC=0.9543  PR-AUC=0.4894  G-mean=0.8211
    LGBM  | AUC=0.9490  PR-AUC=0.4035  G-mean=0.8254
    MLP   | AUC=0.9480  PR-AUC=0.8084  G-mean=0.8928

  === fraud_detection complete; results saved incrementally ===

Dataset : pima_diabetes  |  n_train=537  |  minority=34.82%


...

  Generated: n=2,046  minority=50.00%  [232.2s]
    RF    | AUC=0.7331  PR-AUC=0.1594  G-mean=0.0000
    LGBM  | AUC=0.7357  PR-AUC=0.1890  G-mean=0.0000
    MLP   | AUC=0.6727  PR-AUC=0.1399  G-mean=0.2511

  === secom complete; results saved incrementally ===

Dataset : thyroid_sick  |  n_train=2,640  |  minority=6.14%


ctdGAN Sampling     : 100%|██████████| 2/2 [00:00<00:00,  6.85it/s]


  Generated: n=4,956  minority=50.00%  [116.2s]
    RF    | AUC=0.9939  PR-AUC=0.9342  G-mean=0.8493
    LGBM  | AUC=0.9976  PR-AUC=0.9570  G-mean=0.9515
    MLP   | AUC=0.9583  PR-AUC=0.7069  G-mean=0.8710

  === thyroid_sick complete; results saved incrementally ===

Dataset : unsw_nb15  |  n_train=122,738  |  minority=1.00%


ctdGAN Sampling     : 100%|██████████| 2/2 [00:17<00:00,  8.73s/it]


  Generated: n=243,032  minority=50.00%  [12481.8s]
    RF    | AUC=0.8860  PR-AUC=0.0505  G-mean=0.4426
    LGBM  | AUC=0.9146  PR-AUC=0.1340  G-mean=0.2762
    MLP   | AUC=0.8927  PR-AUC=0.0871  G-mean=0.2049

  === unsw_nb15 complete; results saved incrementally ===

04b experiment complete.
